# Prompt Entropy Degradation — Experiment Notebook

**Semantic Equivalence, Structural Fragility**

Pipeline: load GSM8k → apply perturbations → extract logits → compute Shannon entropy → correlate with accuracy.

## 1. Setup

In [ ]:
import sys
sys.path.append('../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

from perturbations import apply_all_perturbations
from entropy import shannon_entropy, delta_entropy
from model_runner import load_model, extract_logits

## 2. Load Dataset (GSM8k)

In [ ]:
from datasets import load_dataset

gsm8k = load_dataset("gsm8k", "main", split="test")
# TODO: define sample size for the experiment (e.g. 100-200 problems)
sample = gsm8k.select(range(100))

## 3. Generate Perturbed Variants

In [ ]:
# TODO: apply_all_perturbations() to each item in `sample`
# and store the four variants (base, whitespace, markdown, json)

## 4. Load Model

In [ ]:
# Gemma-2 is gated on the Hub: accept the license on the model page and
# run `huggingface-cli login` (or set HF_TOKEN) before this cell works.
# 9B is the size used for the paper's real runs (GPU recommended -- see
# CPU note in src/model_runner.py); 2b is a much faster stand-in for
# local smoke testing on CPU.
MODEL_NAME = "google/gemma-2-9b"  # CPU-only? try "google/gemma-2-2b" instead
model, tokenizer = load_model(MODEL_NAME)

## 5. Extract Logits for Each Variant

In [ ]:
# TODO: loop over the perturbed dataset, call extract_logits()
# for each variant, and store the raw logit arrays

## 6. Compute Shannon Entropy

In [ ]:
# TODO: apply shannon_entropy() to each logit array,
# then delta_entropy() between base and each perturbation type

## 7. Correlate Entropy Delta with Correctness

In [ ]:
# TODO: check final generated answer against GSM8k ground truth,
# then correlate delta_H with correct/incorrect outcome
# (e.g. logistic regression, boxplots by outcome)

## 8. Visualization (for Deliverable 3)

In [ ]:
# TODO: entropy distribution plots per perturbation type
# TODO: delta_H vs. accuracy scatter/boxplot
# TODO: save figures to ../results/figures/

## 9. Sanity Check — Single Example, Full Pipeline

Runs one GSM8k problem through all three perturbations → logit extraction →
entropy, end to end, so you can verify the wiring before scaling to the full
sample. Uses `apply_all_perturbations()` now that `perturb_whitespace`,
`perturb_markdown`, and `perturb_json` are all implemented in
`src/perturbations.py`.

In [ ]:
# Try one real GSM8k problem; fall back to a hardcoded question of the
# same shape if the dataset can't be downloaded (offline / no HF access)
# so this smoke test still runs.
try:
    smoke_example = gsm8k[0] if "gsm8k" in dir() else load_dataset(
        "gsm8k", "main", split="test"
    )[0]
    smoke_question = smoke_example["question"]
except Exception as e:
    print(f"Falling back to a hardcoded example (dataset load failed: {e})")
    smoke_question = (
        "Janet's ducks lay 16 eggs per day. She eats three for breakfast "
        "and bakes muffins with four more. She sells the rest at $2 per "
        "egg. How much money does she make every day?"
    )

K = 5  # number of generated tokens to inspect

variants = apply_all_perturbations(smoke_question)
for name, text in variants.items():
    print(f"{name.upper():>10}: {text[:80]!r}...")

# Mean Shannon entropy (bits) over the first K generated tokens, per variant.
mean_entropy = {}
for name, text in variants.items():
    logits = extract_logits(model, tokenizer, text, k=K)
    entropies = [shannon_entropy(step) for step in logits]
    mean_entropy[name] = float(np.mean(entropies))

print(f"\nMean H over first {K} tokens (bits), and delta vs. base:")
print(f"{'variant':>10}  {'mean_H':>8}  {'delta_H':>8}")
for name, h in mean_entropy.items():
    d = delta_entropy(mean_entropy["base"], h)
    print(f"{name:>10}  {h:8.4f}  {d:+8.4f}")

print("\nIf this ran without errors, the pipeline (perturb -> generate -> "
      "logits -> entropy) is wired correctly end to end for all three "
      "perturbation types.")